# Campus SVI — analysis

Reads the acquisition deliverables and produces tables and publication figures.
**Fetches nothing** — run `01_acquisition.ipynb` first.

| Reads | Writes |
|---|---|
| `data/cells/{campus}_cells.gpkg` | `data/analysis/tables/*.csv` |
| `data/points/{campus}_points.gpkg` | `data/analysis/figures/*.pdf` and `.png` |
| `boundaries/*.gpkg` | `data/reference/campus_registry.csv` |

The grid is **20 m**. Coverage ratios are lower than at a coarser cell by construction — a
smaller cell is harder to intersect — so these numbers are not comparable with results computed
at another cell size. State the size wherever a ratio is reported.

---
## 1 · Setup


In [ ]:
#@title Install
!pip install -q geopandas pyogrio matplotlib
print('ok')


In [ ]:
#@title Mount Drive and load the repo
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/aditpradana36/campus-svi-availability.git'  #@param {type:'string'}
PROJECT_ROOT = '/content/drive/MyDrive/campus-svi-availability'  #@param {type:'string'}

import os, sys
REPO_DIR = '/content/campus-svi-acquisition'
if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from campus_svi import config, registry
config.set_root(PROJECT_ROOT)

from campus_svi.analysis import metrics, figures, maps, style as st
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CAMPUSES = metrics.available()
print(f'{len(CAMPUSES)} campuses with cell data')
print(', '.join(registry.display_names(CAMPUSES)))


---
## 2 · Campus registry

Identity and location for every campus. Centroid, area, perimeter and UTM zone are derived from
your boundary files; display name, city and province come from the lookup in
`campus_svi/registry.py` and are **hand-entered**.

Multi-site universities keep the institution's own designation — UNESA Campus 1 and 2, UNAIR
Campus B and C — rather than place names, since those are the official labels.

Every row is written with `verified = False`. Check the city and province column once, then set
it True. Three abbreviations differ from their slug and are worth a second look: `unbraw` is UB,
`unlam` is ULM, `unud_jimbaran` is UDAYANA.


In [ ]:
#@title Build the registry CSV
reg_path = registry.write(CAMPUSES)
reg = registry.load()

display(reg[['campus_id','display_name','city','province','island',
             'centroid_lat','centroid_lon','area_km2','n_cells']])


---
## 3 · Tables

Numbers first. Every value in a figure is also written as a CSV, so anything surprising in a
plot can be checked against the table rather than read off the pixels.


In [ ]:
#@title Build every table
paths = metrics.write_tables(CAMPUSES)

cov = metrics.coverage_table(CAMPUSES)
display(cov.sort_values('either_coverage', ascending=False)
        [['display_name','n_cells','mly_coverage','ggl_coverage','either_coverage']]
        .round(3).head(12))


In [ ]:
#@title Headline numbers
print(f"campuses         : {len(cov)}")
print(f"grid cells (20 m): {cov['n_cells'].sum():,}")
print(f"Mapillary images : {cov['mly_images'].sum():,}")
print(f"Google panoramas : {cov['ggl_panoramas'].sum():,}")
print()
print(f"mean Mapillary coverage : {cov['mly_coverage'].mean():.1%}")
print(f"mean Google coverage    : {cov['ggl_coverage'].mean():.1%}")
print(f"mean either-source      : {cov['either_coverage'].mean():.1%}")
print()
for k in ('both','mapillary_only','google_only','neither'):
    print(f"  {k:<16} {cov[f'prop_{k}'].mean():.1%}")


---
## 4 · Coverage and agreement


In [ ]:
#@title Figure 1 — coverage by campus
fig, ax = figures.fig_coverage(CAMPUSES)
plt.show()


### On the maps

Each panel is **fitted to its own campus boundary**, so every campus fills its frame regardless
of size. That keeps internal structure legible on a small campus and a large one alike.

Because scale therefore differs between panels, **each panel carries its own scale bar**, placed
below the frame — panels are filled edge to edge, so there is no reliable empty corner inside.
A single shared bar would be false here.

What panel size no longer encodes is campus extent. The bars carry it; set `show_area=True` to
print each campus's area beside its name as well.

At 20 m a campus can carry thousands of cells, so the cell layer is rasterised per panel while
boundaries, text and bars stay vector — identical in print, but a PDF that opens.


In [ ]:
#@title Figure 2 — agreement maps
NCOLS = 8  #@param {type:'integer'}
SHOW_AREA = False  #@param {type:'boolean'}

fig, axes = figures.fig_agreement_maps(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title Figure 2b — agreement composition
fig, ax = figures.fig_agreement_composition(CAMPUSES)
plt.show()


---
## 5 · Enclosure — the depth decay test

Distance from each cell centroid to the campus edge, normalised by that campus's own maximum so
0 is the perimeter and 1 the deepest interior point. Road-free: it needs only the boundary.

The **slope** of coverage against normalised depth is the openness index. Steeply negative means
coverage collapses inward — a more enclosed campus. This does double duty as both the test of
the enclosure hypothesis and the openness measure used to compare campuses.


In [ ]:
#@title Figure 3 — decay curves
HIGHLIGHT = ''  #@param {type:'string'}
hl = [c.strip() for c in HIGHLIGHT.split(',') if c.strip()]

fig, ax = figures.fig_decay(CAMPUSES, highlight=hl)
plt.show()

sl = metrics.decay_slope(CAMPUSES).sort_values('openness_slope')
sl['name'] = registry.display_names(sl['campus_id'])
display(sl[['name','openness_slope','edge_coverage','core_coverage','r2']].round(3).head(10))


In [ ]:
#@title Figure 3b — openness index
fig, ax = figures.fig_openness(CAMPUSES)
plt.show()


---
## 6 · Temporal

**Figure 4** — distinct Google capture years per cell.

**Figure 4b** — Mapillary minus Google capture years where both sources cover the cell.
Diverging ramp centred on zero, a real midpoint here. Cells covered by one source are blank,
since the comparison is undefined there.

**Figure 5** — annual volume by source, the monthly Mapillary series, and burstiness against
contributor concentration.


In [ ]:
#@title Figures 4 and 4b
figures.fig_temporal_depth(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
plt.show()
figures.fig_depth_diff(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
plt.show()


In [ ]:
#@title Figure 5 — temporal signature
fig, axes = figures.fig_temporal_signature(CAMPUSES)
plt.show()

sig = metrics.temporal_signature(CAMPUSES)
sig['name'] = registry.display_names(sig['campus_id'])
display(sig[['name','n_months_active','cv_monthly','top_creator_share','n_sequences']]
        .round(3).head(12))


---
## 7 · Capture programme

Google's `source` field records how each panorama was captured. `launch` is car coverage snapped
to roads; **`scout` is trekker or tripod coverage that is not**; `innerspace` is Business View.

This turns the road-snapping caveat into a measurement: rather than stating that Google coverage
is road-constrained, report the proportion that isn't.


In [ ]:
#@title Figure 6 — capture programme
fig, ax = figures.fig_programme(CAMPUSES)
plt.show()

prog = metrics.programme_table(CAMPUSES)
print(f"mean trekker/tripod share: {prog['scout'].mean():.1%}")
print(f"mean user-upload share   : {prog['third_party'].mean():.1%}")


---
## 8 · Robustness

**MAUP** at 20 / 50 / 100 m, spanning the working resolution. Ratios rise with cell size by
construction, so what matters is whether the campus ranking and the between-source gap survive.
Cheap here: the point deliverable is resolution-independent, so re-gridding needs no refetching.

**Moran's I** because adjacent cells are not independent — without it, any cell-level
significance claim is unsupported. Computed on the lattice from row/column adjacency with a
permutation test.


In [ ]:
#@title Figure S1 — MAUP
SIZES = [20, 50, 100]  #@param
fig, axes = figures.fig_maup(CAMPUSES, sizes=tuple(SIZES))
plt.show()


In [ ]:
#@title Figure S2 — Moran's I
fig, ax = figures.fig_autocorrelation(CAMPUSES)
plt.show()

ac = metrics.autocorrelation_table(CAMPUSES)
sig_n = len(ac[(ac['column']=='either_coverage') & (ac['p_sim']<0.05)])
print(f"{sig_n}/{ac['campus_id'].nunique()} campuses show significant clustering")


---
## 9 · Build everything


In [ ]:
#@title All tables and figures
registry.write(CAMPUSES)
metrics.write_tables(CAMPUSES)
figures.build_all(CAMPUSES, ncols=NCOLS, show_area=SHOW_AREA)
print(f"\noutputs under {config.DATA_DIR / 'analysis'}")


---
## Notes

**Cell size.** 20 m. Report it wherever a coverage ratio appears; the numbers do not transfer
across cell sizes.

**Scope of inference.** Cell-level statistics within a campus rest on thousands of cells at 20 m
and are fine. Across campuses, n = 40 — descriptive typology, not regression.

**Date precision is not uniform.** Official Google coverage is month-level, third-party uploads
can carry a day, Mapillary timestamps are millisecond-precise. Bin to year, or at finest month.

**Restyling.** Colours in `analysis/style.py`, house style in `analysis/paperstyle.py`, names in
`registry.py`. Change them there and every figure follows.
